# NB01: Genome and Carbon Source Selection

**Project**: Annotation-Gap Discovery
**Goal**: Select ~20 Fitness Browser organisms with rich carbon-source experiment coverage, cross-reference with BacDive growth phenotypes, and map to BERDL pangenome.

**Requires**: BERDL JupyterHub (Spark session)

## Outputs
- `data/genome_manifest.tsv` — selected genomes with metadata
- `data/carbon_source_panel.tsv` — carbon source panel with mapping IDs
- `data/fb_carbon_experiments.tsv` — all FB carbon source experiments
- `data/bacdive_utilization.tsv` — BacDive growth data for matched organisms

In [ ]:
# Initialize Spark session (BERDL JupyterHub — no import needed)
spark = get_spark_session()
spark.sql('SELECT 1 AS ok').show()
print('Spark session ready.')

In [ ]:
import pandas as pd
import numpy as np
import os

DATA_DIR = '../data'
os.makedirs(DATA_DIR, exist_ok=True)

## 1. Fitness Browser: Carbon Source Experiments

Query the `experiment` table for `expGroup = 'carbon source'` to find which organisms have carbon-source fitness data and how many distinct conditions each has.

In [ ]:
# Get all carbon source experiments from Fitness Browser
fb_carbon_exps = spark.sql("""
    SELECT orgId, expName, expGroup, condition_1, concentration_1, units_1,
           condition_2, media
    FROM kescience_fitnessbrowser.experiment
    WHERE expGroup = 'carbon source'
""").toPandas()

print(f'Total carbon source experiments: {len(fb_carbon_exps)}')
print(f'Organisms with carbon source data: {fb_carbon_exps["orgId"].nunique()}')
print(f'Distinct carbon sources: {fb_carbon_exps["condition_1"].nunique()}')
fb_carbon_exps.head(10)

In [ ]:
# Per-organism summary: how many distinct carbon sources and experiments
fb_org_summary = fb_carbon_exps.groupby('orgId').agg(
    n_experiments=('expName', 'nunique'),
    n_carbon_sources=('condition_1', 'nunique'),
    carbon_sources=('condition_1', lambda x: ', '.join(sorted(x.unique())[:10]))
).reset_index().sort_values('n_carbon_sources', ascending=False)

print(f'Organisms with >= 10 carbon sources: {(fb_org_summary["n_carbon_sources"] >= 10).sum()}')
fb_org_summary.head(30)

## 2. Fitness Browser: Organism Metadata

Get taxonomy, genome info, and gene counts for all FB organisms.

In [ ]:
# Get organism metadata
fb_organisms = spark.sql("""
    SELECT orgId, genus, species, strain, taxonomyId, 
           genome_name, nGenesUsed
    FROM kescience_fitnessbrowser.organism
""").toPandas()

# Also get total experiment counts per organism (all conditions)
fb_all_exps = spark.sql("""
    SELECT orgId, expGroup, COUNT(*) as n_exps
    FROM kescience_fitnessbrowser.experiment
    GROUP BY orgId, expGroup
""").toPandas()

print(f'Total FB organisms: {len(fb_organisms)}')
fb_organisms.head(10)

In [ ]:
# Merge organism metadata with carbon source summary
fb_merged = fb_org_summary.merge(fb_organisms, on='orgId', how='inner')
fb_merged['full_name'] = fb_merged['genus'] + ' ' + fb_merged['species'] + ' ' + fb_merged['strain'].fillna('')
fb_merged = fb_merged.sort_values('n_carbon_sources', ascending=False)

print(f'FB organisms with carbon source experiments: {len(fb_merged)}')
print(f'\nTop 20 by carbon source count:')
fb_merged[['orgId', 'full_name', 'n_carbon_sources', 'n_experiments', 'taxonomyId']].head(20)

## 3. Experiment Group Distribution

See what other condition types are available for these organisms.

In [ ]:
# Experiment group distribution across all FB organisms
exp_group_dist = fb_all_exps.groupby('expGroup')['n_exps'].sum().sort_values(ascending=False)
print('Experiment types across all FB organisms:')
print(exp_group_dist.to_string())

## 4. BacDive: Growth Phenotype Data

Query BacDive metabolite utilization for organisms that match our FB candidates.
BacDive stores +/- growth calls per strain per compound.

In [ ]:
# First check what tables BacDive has
spark.sql('SHOW TABLES IN kescience_bacdive').show(truncate=False)

In [ ]:
# Check BacDive metabolite_utilization schema
spark.sql('DESCRIBE kescience_bacdive.metabolite_utilization').show(truncate=False)

In [ ]:
# Sample BacDive metabolite_utilization data
spark.sql("""
    SELECT * FROM kescience_bacdive.metabolite_utilization LIMIT 10
""").show(truncate=False)

In [ ]:
# Get BacDive utilization summary — how many compounds tested per genus/species
# BacDive uses NCBI taxonomy names, FB uses its own orgIds
# We'll need to match by genus+species

# First, get the distinct genera in our FB candidate list
fb_genera = fb_merged['genus'].unique().tolist()
genera_str = "', '".join(fb_genera)

print(f'FB genera to match in BacDive: {len(fb_genera)}')
print(fb_genera)

In [ ]:
# Check BacDive strain table for taxonomy info
spark.sql('DESCRIBE kescience_bacdive.strain').show(truncate=False)

In [ ]:
# Get BacDive strains that match our FB genera
# BacDive has species_epithet, genus columns in strain table
bacdive_strains = spark.sql(f"""
    SELECT strain_id, genus, species_epithet, ncbi_tax_id,
           full_scientific_name
    FROM kescience_bacdive.strain
    WHERE genus IN ('{genera_str}')
""").toPandas()

print(f'BacDive strains matching FB genera: {len(bacdive_strains)}')
print(f'BacDive genera matched: {bacdive_strains["genus"].nunique()}')
bacdive_strains.groupby('genus').size().sort_values(ascending=False).head(20)

In [ ]:
# Get metabolite utilization data for matched strains
if len(bacdive_strains) > 0:
    matched_strain_ids = bacdive_strains['strain_id'].tolist()
    
    # Register as temp view for efficient join
    spark.createDataFrame(
        pd.DataFrame({'strain_id': matched_strain_ids})
    ).createOrReplaceTempView('matched_strains')
    
    bacdive_util = spark.sql("""
        SELECT mu.strain_id, mu.compound, mu.utilization, mu.chebi_id,
               mu.kind
        FROM kescience_bacdive.metabolite_utilization mu
        JOIN matched_strains ms ON mu.strain_id = ms.strain_id
    """).toPandas()
    
    print(f'BacDive utilization records for matched strains: {len(bacdive_util)}')
    print(f'Distinct compounds: {bacdive_util["compound"].nunique()}')
    print(f'\nUtilization values:')
    print(bacdive_util['utilization'].value_counts())
else:
    print('No BacDive strains matched — will proceed with FB data only')
    bacdive_util = pd.DataFrame()

In [ ]:
# Merge BacDive utilization with strain metadata to get genus/species
if len(bacdive_util) > 0:
    bacdive_util = bacdive_util.merge(
        bacdive_strains[['strain_id', 'genus', 'species_epithet']],
        on='strain_id', how='left'
    )
    
    # Summarize: per genus, how many compounds have +/- data
    bacdive_genus_summary = bacdive_util.groupby('genus').agg(
        n_compounds=('compound', 'nunique'),
        n_positive=('utilization', lambda x: (x == '+').sum()),
        n_negative=('utilization', lambda x: (x == '-').sum()),
        n_records=('utilization', 'count')
    ).sort_values('n_compounds', ascending=False)
    
    print('BacDive compound coverage per genus (matched to FB):')
    bacdive_genus_summary.head(20)

## 5. Map FB Organisms to Pangenome

Link Fitness Browser organisms to BERDL pangenome species clades.
Use the existing `fb_pangenome_link.tsv` from `conservation_vs_fitness` if available,
otherwise match via taxonomy.

In [ ]:
# Try to load existing FB-pangenome link table
link_path = '../../conservation_vs_fitness/data/fb_pangenome_link.tsv'
if os.path.exists(link_path):
    fb_pg_link = pd.read_csv(link_path, sep='\t')
    print(f'Loaded existing FB-pangenome link table: {len(fb_pg_link)} links')
    print(f'Organisms covered: {fb_pg_link["orgId"].nunique() if "orgId" in fb_pg_link.columns else "unknown"}')
    fb_pg_link.head()
else:
    print(f'Link table not found at {link_path}')
    print('Will match via taxonomy instead')
    fb_pg_link = None

In [ ]:
# Match FB organisms to pangenome species clades via taxonomy
# Strategy: use genus + species to find matching GTDB species clades

# Get FB organism genus+species for matching
fb_for_match = fb_merged[['orgId', 'genus', 'species', 'strain', 'taxonomyId', 'full_name']].copy()

# Query GTDB taxonomy for matching species
# GTDB species names use format s__Genus_species
gtdb_species = spark.sql("""
    SELECT DISTINCT gtdb_species_clade_id, GTDB_species
    FROM kbase_ke_pangenome.gtdb_species_clade
""").toPandas()

print(f'Total GTDB species clades: {len(gtdb_species)}')

# Build a lookup: genus_species -> gtdb_species_clade_id
# GTDB format: s__Genus_species (possibly with suffix like _A, _B)
gtdb_species['genus_species'] = gtdb_species['GTDB_species'].str.replace('s__', '').str.replace('_', ' ')

# Match FB organisms
matches = []
for _, row in fb_for_match.iterrows():
    fb_name = f"{row['genus']} {row['species']}"
    # Try exact match first
    exact = gtdb_species[gtdb_species['genus_species'] == fb_name]
    if len(exact) > 0:
        for _, m in exact.iterrows():
            matches.append({
                'orgId': row['orgId'],
                'fb_name': fb_name,
                'gtdb_species_clade_id': m['gtdb_species_clade_id'],
                'GTDB_species': m['GTDB_species'],
                'match_type': 'exact'
            })
    else:
        # Try genus-level match with GTDB suffixes (_A, _B, etc.)
        partial = gtdb_species[gtdb_species['genus_species'].str.startswith(fb_name)]
        if len(partial) > 0:
            for _, m in partial.iterrows():
                matches.append({
                    'orgId': row['orgId'],
                    'fb_name': fb_name,
                    'gtdb_species_clade_id': m['gtdb_species_clade_id'],
                    'GTDB_species': m['GTDB_species'],
                    'match_type': 'partial'
                })
        else:
            matches.append({
                'orgId': row['orgId'],
                'fb_name': fb_name,
                'gtdb_species_clade_id': None,
                'GTDB_species': None,
                'match_type': 'no_match'
            })

fb_gtdb_matches = pd.DataFrame(matches)
print(f'\nMatch results:')
print(fb_gtdb_matches['match_type'].value_counts())
print(f'\nUnmatched organisms:')
print(fb_gtdb_matches[fb_gtdb_matches['match_type'] == 'no_match'][['orgId', 'fb_name']])

In [ ]:
# For matched species, get pangenome stats (genome count, gene clusters)
matched_clades = fb_gtdb_matches[fb_gtdb_matches['gtdb_species_clade_id'].notna()]['gtdb_species_clade_id'].unique()

if len(matched_clades) > 0:
    clade_list = "', '".join(matched_clades)
    pangenome_stats = spark.sql(f"""
        SELECT gtdb_species_clade_id, 
               CAST(no_genomes AS INT) as no_genomes,
               CAST(no_gene_clusters AS INT) as no_gene_clusters,
               CAST(no_core AS INT) as no_core,
               CAST(no_aux_genome AS INT) as no_aux
        FROM kbase_ke_pangenome.pangenome
        WHERE gtdb_species_clade_id IN ('{clade_list}')
    """).toPandas()
    
    print(f'Pangenome stats for {len(pangenome_stats)} matched clades:')
    pangenome_stats.head(20)
else:
    print('No matched clades found')
    pangenome_stats = pd.DataFrame()

## 6. GapMind Pathway Coverage

Check GapMind carbon source pathway predictions for our candidate genomes.

In [ ]:
# Get a representative genome_id for each matched species clade
if len(matched_clades) > 0:
    rep_genomes = spark.sql(f"""
        SELECT gtdb_species_clade_id, genome_id
        FROM kbase_ke_pangenome.genome
        WHERE gtdb_species_clade_id IN ('{clade_list}')
    """).toPandas()
    
    # Take one representative per clade
    rep_genomes_dedup = rep_genomes.groupby('gtdb_species_clade_id').first().reset_index()
    rep_genome_ids = rep_genomes_dedup['genome_id'].tolist()
    
    # GapMind genome IDs lack RS_/GB_ prefix
    gapmind_ids = [g.replace('RS_', '').replace('GB_', '') for g in rep_genome_ids]
    gapmind_ids_str = "', '".join(gapmind_ids)
    
    gapmind_summary = spark.sql(f"""
        SELECT genome_id, metabolic_category, 
               COUNT(DISTINCT pathway) as n_pathways,
               SUM(CASE WHEN score_category = 'complete' THEN 1 ELSE 0 END) as n_complete
        FROM kbase_ke_pangenome.gapmind_pathways
        WHERE genome_id IN ('{gapmind_ids_str}')
        GROUP BY genome_id, metabolic_category
    """).toPandas()
    
    print(f'GapMind data for {gapmind_summary["genome_id"].nunique()} genomes:')
    gapmind_summary.head(20)
else:
    gapmind_summary = pd.DataFrame()

## 7. Select Top 20 Genomes

Rank and select genomes based on:
1. Number of carbon source experiments (more = better)
2. Has BacDive growth data (bonus)
3. Has pangenome match (required)
4. Taxonomic diversity (preferred)

In [ ]:
# Build the selection table
selection = fb_merged.copy()

# Add pangenome match info (take first match per orgId)
first_match = fb_gtdb_matches.drop_duplicates('orgId')
selection = selection.merge(
    first_match[['orgId', 'gtdb_species_clade_id', 'GTDB_species', 'match_type']],
    on='orgId', how='left'
)

# Add pangenome stats
if len(pangenome_stats) > 0:
    selection = selection.merge(
        pangenome_stats[['gtdb_species_clade_id', 'no_genomes', 'no_gene_clusters']],
        on='gtdb_species_clade_id', how='left'
    )

# Add BacDive flag
if len(bacdive_util) > 0:
    bacdive_genera = set(bacdive_genus_summary.index)
    selection['has_bacdive'] = selection['genus'].isin(bacdive_genera)
else:
    selection['has_bacdive'] = False

# Filter: must have pangenome match and >= 10 carbon sources
eligible = selection[
    (selection['gtdb_species_clade_id'].notna()) &
    (selection['n_carbon_sources'] >= 10)
].copy()

# Score: weighted by carbon sources + BacDive bonus
eligible['score'] = eligible['n_carbon_sources'] + eligible['has_bacdive'].astype(int) * 5
eligible = eligible.sort_values('score', ascending=False)

print(f'Eligible organisms (>= 10 C-sources + pangenome match): {len(eligible)}')
eligible[['orgId', 'full_name', 'n_carbon_sources', 'n_experiments', 
          'has_bacdive', 'gtdb_species_clade_id', 'score']].head(25)

In [ ]:
# Select top 20
TARGET_N = 20
selected = eligible.head(TARGET_N).copy()

print(f'Selected {len(selected)} genomes')
print(f'  With BacDive data: {selected["has_bacdive"].sum()}')
print(f'  Mean carbon sources: {selected["n_carbon_sources"].mean():.1f}')
print(f'  Distinct genera: {selected["genus"].nunique()}')
print(f'\nTaxonomic distribution:')
print(selected['genus'].value_counts().to_string())
print(f'\nSelected organisms:')
selected[['orgId', 'full_name', 'n_carbon_sources', 'has_bacdive', 'gtdb_species_clade_id']]

## 8. Carbon Source Panel

Build the carbon source panel — union of all carbon sources across selected genomes.

In [ ]:
# Get carbon source experiments for selected organisms only
selected_orgIds = selected['orgId'].tolist()
selected_exps = fb_carbon_exps[fb_carbon_exps['orgId'].isin(selected_orgIds)].copy()

# Build carbon source panel
carbon_panel = selected_exps.groupby('condition_1').agg(
    n_organisms=('orgId', 'nunique'),
    n_experiments=('expName', 'nunique'),
    organisms=('orgId', lambda x: ', '.join(sorted(x.unique())))
).reset_index().sort_values('n_organisms', ascending=False)

carbon_panel.rename(columns={'condition_1': 'carbon_source'}, inplace=True)

print(f'Total carbon sources across selected organisms: {len(carbon_panel)}')
print(f'Carbon sources tested in >= 5 organisms: {(carbon_panel["n_organisms"] >= 5).sum()}')
print(f'Carbon sources tested in >= 10 organisms: {(carbon_panel["n_organisms"] >= 10).sum()}')
carbon_panel.head(30)

In [ ]:
# Create coverage matrix: organisms × carbon sources (top 30)
top_carbons = carbon_panel.head(30)['carbon_source'].tolist()
coverage_matrix = pd.crosstab(
    selected_exps[selected_exps['condition_1'].isin(top_carbons)]['orgId'],
    selected_exps[selected_exps['condition_1'].isin(top_carbons)]['condition_1']
).clip(upper=1)  # binary presence

print(f'Coverage matrix: {coverage_matrix.shape[0]} organisms × {coverage_matrix.shape[1]} carbon sources')
print(f'Mean coverage per organism: {coverage_matrix.sum(axis=1).mean():.1f} carbon sources')
print(f'Mean coverage per carbon source: {coverage_matrix.sum(axis=0).mean():.1f} organisms')
coverage_matrix

## 9. Extract Fitness Data Summary for Selected Organisms

Get gene count and fitness data availability for each selected organism.

In [ ]:
# Get gene counts and fitness data stats for selected organisms
selected_org_str = "', '".join(selected_orgIds)

fb_gene_counts = spark.sql(f"""
    SELECT orgId, 
           COUNT(*) as n_genes,
           SUM(CASE WHEN type = '1' THEN 1 ELSE 0 END) as n_protein_coding
    FROM kescience_fitnessbrowser.gene
    WHERE orgId IN ('{selected_org_str}')
    GROUP BY orgId
""").toPandas()

fb_fitness_counts = spark.sql(f"""
    SELECT orgId,
           COUNT(DISTINCT locusId) as n_genes_with_fitness,
           COUNT(DISTINCT expName) as n_fitness_experiments
    FROM kescience_fitnessbrowser.genefitness
    WHERE orgId IN ('{selected_org_str}')
    GROUP BY orgId
""").toPandas()

gene_stats = fb_gene_counts.merge(fb_fitness_counts, on='orgId', how='left')
gene_stats['pct_with_fitness'] = (
    gene_stats['n_genes_with_fitness'] / gene_stats['n_protein_coding'] * 100
).round(1)

print('Gene and fitness data summary for selected organisms:')
gene_stats

## 10. Save Outputs

In [ ]:
# Genome manifest
manifest = selected.merge(gene_stats, on='orgId', how='left')
manifest_cols = [
    'orgId', 'full_name', 'genus', 'species', 'strain', 'taxonomyId',
    'n_carbon_sources', 'n_experiments', 'has_bacdive',
    'gtdb_species_clade_id', 'GTDB_species',
    'n_genes', 'n_protein_coding', 'n_genes_with_fitness', 'pct_with_fitness'
]
# Only include columns that exist
manifest_cols = [c for c in manifest_cols if c in manifest.columns]
manifest = manifest[manifest_cols]

manifest.to_csv(f'{DATA_DIR}/genome_manifest.tsv', sep='\t', index=False)
print(f'Saved genome manifest: {len(manifest)} genomes')

# Carbon source panel
carbon_panel.to_csv(f'{DATA_DIR}/carbon_source_panel.tsv', sep='\t', index=False)
print(f'Saved carbon source panel: {len(carbon_panel)} carbon sources')

# Full carbon experiments for selected organisms
selected_exps.to_csv(f'{DATA_DIR}/fb_carbon_experiments.tsv', sep='\t', index=False)
print(f'Saved FB carbon experiments: {len(selected_exps)} experiments')

# BacDive utilization (if available)
if len(bacdive_util) > 0:
    # Filter to genera of selected organisms
    selected_genera = selected['genus'].unique()
    bacdive_selected = bacdive_util[bacdive_util['genus'].isin(selected_genera)]
    bacdive_selected.to_csv(f'{DATA_DIR}/bacdive_utilization.tsv', sep='\t', index=False)
    print(f'Saved BacDive utilization: {len(bacdive_selected)} records')

print('\nAll NB01 outputs saved to data/')

In [ ]:
# Final summary
print('=' * 60)
print('NB01 SUMMARY: Genome and Carbon Source Selection')
print('=' * 60)
print(f'Selected genomes:           {len(selected)}')
print(f'With BacDive data:          {selected["has_bacdive"].sum()}')
print(f'Distinct genera:            {selected["genus"].nunique()}')
print(f'Total carbon sources:       {len(carbon_panel)}')
print(f'Carbon sources (>= 5 orgs): {(carbon_panel["n_organisms"] >= 5).sum()}')
print(f'Mean C-sources per org:     {selected["n_carbon_sources"].mean():.1f}')
print(f'Mean experiments per org:    {selected["n_experiments"].mean():.1f}')
print('=' * 60)